In [1]:
import sys
import os

sys.path.append(os.path.abspath("../.."))

from config.spark import spark


In [2]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,TimestampType,DoubleType

order_items_schema = StructType([
    StructField("order_id",StringType()),
    StructField("order_item_id",IntegerType()),
    StructField("product_id",StringType()),
    StructField("seller_id",StringType()),
    StructField("shipping_limit_date",TimestampType()),
    StructField("price",DoubleType()),
    StructField("freight_value",DoubleType()),
])


In [3]:
order_items_df = (
    spark.read.format('csv')
    .option('header','True')
   # .option('inferschema','True')
   .schema(order_items_schema)
    .load('../../landing/files/olist_order_items_dataset.csv')
    
)

In [4]:
from pyspark.sql.functions import current_timestamp

order_items_df = order_items_df.withColumn("ingestion_time",current_timestamp())

In [5]:
order_items_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)



In [6]:
order_items_df.write.mode("overwrite").saveAsTable("bronze.order_items")

In [7]:
spark.sql(
    """
    select * from bronze.order_items
    """
).show(100)

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+--------------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|      ingestion_time|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+--------------------+
|89bed55dd88d035e3...|            1|ae6b739ab6e9d7991...|53e4c6e0f4312d4d2...|2018-01-11 17:12:22|  42.0|         17.6|2026-10-19 15:28:...|
|89c037e2b749a2ed5...|            1|e906fa76a27488f80...|1835b56ce799e6a4d...|2017-11-30 04:15:33| 53.99|        12.72|2026-10-19 15:28:...|
|89c037e2b749a2ed5...|            2|e906fa76a27488f80...|1835b56ce799e6a4d...|2017-11-30 04:15:33| 53.99|        12.72|2026-10-19 15:28:...|
|89c04d22504649482...|            1|3d73c88390adac7dd...|c8b0e2b0a7095e5d8...|2018-03-14 00:20:26|199.99|        25.05|2026-10-19 15:28:...|
|89c0bf5292a4